## Распознавание именованных сущностей с использованием BERT

Основано на блокноте https://github.com/huggingface/notebooks/blob/master/examples/token_classification.ipynb

BERT (англ. Bidirectional Encoder Representations from Transformers – двунаправленный кодировщик представлений трансформера) — языковая модель, основанная на архитектуре Трансформер (Transformer), предназначенная для предобучения *языковых представлений* (Representation) с целью их последующего применения в широком спектре задач обработки естественного языка (NLP).

## Как работает BERT
BERT использует Трансформер – архитектуру, которая изучает контекстуальные отношения между словами в тексте. Она включает в себя два отдельных механизма — кодировщик, считывающий введенный текст, и декодер, выдающий прогноз. Поскольку целью BERT является создание языковой модели, он использует только механизм кодировщика.

В отличие от однонаправленных моделей, которые считывают вводимый текст последовательно (слева направо или справа налево), трансформер считывает сразу всю последовательность слов. Поэтому он считается двунаправленным, хотя правильнее было бы сказать, ненаправленным. Это свойство позволяет модели изучать контекст слова на основе всего его окружения.

Принцип работы трансформера представлен на рисунке. Вход представляет собой последовательность токенов (Token) (слов, их частей или символов), которые сначала преобразуются в векторы, а затем обрабатываются нейронной сетью. Выход представляет собой последовательность векторов, в которой каждый вектор соответствует входной лексеме с тем же индексом.
![alt text](https://resize.yandex.net/mailservice?url=https%3A%2F%2Fwww.helenkapatsa.ru%2Fcontent%2Fimages%2F2022%2F08%2Fimage.png&proxy=yes&key=91768e122e301805e3be976cd66cbc7d)

В этом блокноте модель дообучается на задаче классификации отдельных слов, а именно решается задача *распознавания именованных сущностей* (named entity recognition, *NER*). Используется датасет медицинских сущностей, но в целом пайплайн подходит для любой задачи на выделение сущностей в тексте.

In [1]:
# Устанавливаем необходимые зависимости
# seqeval - фреймворк для оценки маркировки последовательности,
# используется для оценки эффективности в задачах разбиения на фрагменты, таких как распознавание именованных сущностей, разметка частей речи и др.

! pip install -q datasets transformers seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


Для скорости используем маленький BERT для русского языка [rubert-tiny](https://huggingface.co/cointegrated/rubert-tiny). Если взять другую, более крупную BERT-подобную модель, качество NER может быть выше, но и время обучения и работы будет дольше.


Этот ноутбук может быть использован для любой задачи классификации токенов с любой моделью из [Model Hub](https://huggingface.co/models), если у этой модели есть версия для классификации токенов с быстрым токенизатором (это можно проверить в [таблице](https://huggingface.co/transformers/index.html#bigtable)). Возможно, потребуются небольшие корректировки при использовании других наборов данных. В зависимости от модели и используемого графического процессора может потребоваться настроить размер пакета, чтобы избежать ошибок нехватки памяти.

In [2]:
model_checkpoint = "cointegrated/rubert-tiny"
batch_size = 16

## Загрузка данных

Для обучения возьмём размеченный корпус русскоязычных отзывов на лекарства [Russian Drug Reaction Corpus](https://github.com/cimm-kzn/RuDReC).

Загрузим его библиотекой corus, потому что это удобно

In [3]:
from datasets import load_dataset

In [4]:
!wget https://github.com/cimm-kzn/RuDReC/raw/master/data/rudrec_annotated.json
!pip install -q corus razdel

--2025-12-18 09:20:13--  https://github.com/cimm-kzn/RuDReC/raw/master/data/rudrec_annotated.json
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/cimm-kzn/RuDReC/master/data/rudrec_annotated.json [following]
--2025-12-18 09:20:14--  https://raw.githubusercontent.com/cimm-kzn/RuDReC/master/data/rudrec_annotated.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1773014 (1.7M) [text/plain]
Saving to: ‘rudrec_annotated.json’

rudrec_annotated.js 100%[===================>]   1.69M  --.-KB/s    in 0.06s   

2025-12-18 09:20:14 (26.1 MB/s) - ‘rudrec_annotated.json’ saved [1773014/1773014]

   ━━━━━━

In [5]:
from corus import load_rudrec
drugs = list(load_rudrec('rudrec_annotated.json'))
print(len(drugs))

4809


Пример документа:

In [6]:
drugs[0]

RuDReCRecord(
    file_name='172744.tsv',
    text='нам прописали, так мой ребенок сыпью покрылся, глаза опухли, сверху и снизу на веках высыпала сыпь, ( 8 месяцев сыну)А от виферона такого не было... У кого ещё такие побочки, отзовитесь!1 Чем спасались?\n',
    sentence_id=0,
    entities=[RuDReCEntity(
         entity_id='*[0]_se',
         entity_text='виферона',
         entity_type='Drugform',
         start=122,
         end=130,
         concept_id='C0021735',
         concept_name=nan
     ),
     RuDReCEntity(
         entity_id='*[1]',
         entity_text='сыпью покрылся',
         entity_type='ADR',
         start=31,
         end=45,
         concept_id='C0015230',
         concept_name=nan
     ),
     RuDReCEntity(
         entity_id='*[2]',
         entity_text='глаза опухли',
         entity_type='ADR',
         start=47,
         end=59,
         concept_id='C4760994',
         concept_name=nan
     ),
     RuDReCEntity(
         entity_id='*[3]',
         entity_text

Посмотрим, какие сущности есть: лекарства, форма лекарств, класс лекарств, показания к применению, побочные действия и прочие болезни/симптомы.

https://arxiv.org/abs/2004.03659

* **DRUGNAME** - упоминание торговой марки препарата или ингредиентов/активных соединений продукта.
* **DRUGCLASS** - упоминание класса препарата, например, противовоспалительные или сердечно-сосудистые.
* **DRUGFORM** - упоминание о способах введения, например, таблетка или жидкость, которые описывают физическую форму, в которой лекарство будет доставлено в организм пациента.
* **DI** - любое указание/симптом, указывающий на причину
прием/назначение препарата.
* **ADR** - упоминание неблагоприятных побочных действий, которые происходят в результате приема лекарств и не связаны с симптомами лечения.
* **FINDING** - любой DI или ADR, которые не были непосредственно испытаны пациентом или членами его/ее семьи, или связаны с историей болезни/этикеткой препарата или любыми формами болезни, если аннотатор не ясно указал тип

In [7]:
# Сollections - встроенный модуль Python для работы с данными контейнеронного типа
# Счетчик Counter позволяет вычислить частоту вхождений каждого элемента
# Возвращет словарь с элементами в качестве ключей и "счетчиком" (количеством вхождений элемента) в качестве значений
# most_common(n) возвращает в порядке убывания список n наиболее часто встречающихся элементов с указанием их количества
# defaultdict при обращении к отсутствующему ключу возвращает значение по умолчанию
# в остальном он ничем не отличается от обычного словаря

from collections import Counter, defaultdict
type2text = defaultdict(Counter)
ents = Counter()
for item in drugs:
    for e in item.entities:
        ents[e.entity_type] += 1
        type2text[e.entity_type][e.entity_text] += 1

for k, v in ents.most_common():
    print(k, v)
    print(type2text[k].most_common(3))

DI 1401
[('простуды', 64), ('ОРВИ', 47), ('профилактики', 42)]
Drugname 1043
[('Виферон', 33), ('Анаферон', 25), ('Циклоферон', 24)]
Drugform 836
[('таблетки', 154), ('таблеток', 79), ('свечи', 63)]
ADR 720
[('аллергия', 16), ('слабость', 13), ('диарея', 12)]
Drugclass 330
[('противовирусный', 21), ('противовирусное', 18), ('противовирусных', 13)]
Finding 236
[('аллергии', 12), ('температуры', 6), ('сонливости', 5)]


In [8]:
drugs[0].text

'нам прописали, так мой ребенок сыпью покрылся, глаза опухли, сверху и снизу на веках высыпала сыпь, ( 8 месяцев сыну)А от виферона такого не было... У кого ещё такие побочки, отзовитесь!1 Чем спасались?\n'

Напишем функцию, перекладывающую разметку сущностей на уровень слов. Будем использовать [IOB](https://en.wikipedia.org/wiki/Inside–outside–beginning_(tagging))-нотацию, чтобы разделять несколько сущностей одного типа, идущих подряд.

In [9]:
from razdel import tokenize

def extract_labels(item):
    raw_toks = list(tokenize(item.text))
    words = [tok.text for tok in raw_toks]
    word_labels = ['O'] * len(raw_toks)
    char2word = [None] * len(item.text)
    for i, word in enumerate(raw_toks):
        char2word[word.start:word.stop] = [i] * len(word.text)

    for e in item.entities:
        e_words = sorted({idx for idx in char2word[e.start:e.end] if idx is not None})
        word_labels[e_words[0]] = 'B-' + e.entity_type
        for idx in e_words[1:]:
            word_labels[idx] = 'I-' + e.entity_type

    return {'tokens': words, 'tags': word_labels}

In [10]:
print(extract_labels(drugs[0]))

{'tokens': ['нам', 'прописали', ',', 'так', 'мой', 'ребенок', 'сыпью', 'покрылся', ',', 'глаза', 'опухли', ',', 'сверху', 'и', 'снизу', 'на', 'веках', 'высыпала', 'сыпь', ',', '(', '8', 'месяцев', 'сыну', ')', 'А', 'от', 'виферона', 'такого', 'не', 'было', '...', 'У', 'кого', 'ещё', 'такие', 'побочки', ',', 'отзовитесь', '!', '1', 'Чем', 'спасались', '?'], 'tags': ['O', 'O', 'O', 'O', 'O', 'O', 'B-ADR', 'I-ADR', 'O', 'B-ADR', 'I-ADR', 'O', 'O', 'O', 'O', 'B-ADR', 'I-ADR', 'I-ADR', 'I-ADR', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Drugform', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}


In [11]:
from sklearn.model_selection import train_test_split
ner_data = [extract_labels(item) for item in drugs]
ner_train, ner_test = train_test_split(ner_data, test_size=0.2, random_state=1)

Пример данных

In [12]:
import pandas as pd
pd.options.display.max_colwidth = 300
pd.DataFrame(ner_train).sample(3)

,tokens,tags
1791,"[Кроме, "", Ингавирина, "", я, пью, чай, травяной, с, медом, .]","[O, O, B-Drugname, O, O, O, O, O, O, O, O]"
0,"[По, мне, ,, так, это, пустышка, ,, которая, может, обладать, только, эффектом, плацебо, .]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]"
1181,"[Я, этот, препарат, в, основном, принимал, на, вечер, ,, так, как, от, Барбовала, идёт, спецефичесский, запах, ,, за, рулём, употребляя, данный, препарат, ездить, нельзя, ,, у, меня, нарушались, функции, движения, и, координации, ,, такое, впечатление, ,, что, я, был, пьяный, ,, а, в, таком, сос...","[O, O, O, O, O, O, O, O, O, O, O, O, B-Drugname, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-ADR, I-ADR, I-ADR, I-ADR, I-ADR, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]"


Соберём все виды меток в список.

In [13]:
label_list = sorted({label for item in ner_train for label in item['tags']})
if 'O' in label_list:
    label_list.remove('O')
    label_list = ['O'] + label_list
label_list

['O',
 'B-ADR',
 'B-DI',
 'B-Drugclass',
 'B-Drugform',
 'B-Drugname',
 'B-Finding',
 'I-ADR',
 'I-DI',
 'I-Drugclass',
 'I-Drugform',
 'I-Drugname',
 'I-Finding']

Сложим наши данные в объект [`DatasetDict`](https://huggingface.co/docs/datasets/package_reference/main_classes.html#datasetdict), нативный для huggingface.

In [14]:
from datasets import Dataset, DatasetDict

In [15]:
ner_data = DatasetDict({
    'train': Dataset.from_pandas(pd.DataFrame(ner_train)),
    'test': Dataset.from_pandas(pd.DataFrame(ner_test))
})
ner_data

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 3847
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 962
    })
})

## Preprocessing the data

Прежде чем мы сможем найти эти тексты для нашей модели, нам
нужно их предварительно обработать. Это делается с помощью токенизатора Transformers, который (как следует из названия) маркирует входные данные
(включая преобразование токенов в их соответствующие идентификаторы в
предварительно подготовленном словаре) и помещает их в формат,
ожидаемый моделью, а также генерирует другие входные данные, которые
требуются модели.

Чтобы это сделать, мы создадим экземпляр нашего токенизатора с
помощью метода AutoTokenizer.from_pretrained, который обеспечит:

- мы получаем токенизатор, соответствующий архитектуре модели,
которую мы хотим использовать,
- мы загружаем словарь, используемый при предварительном обучении
этой конкретной контрольной точке.

Этот словарь будет вызван, поэтому он не будет загружен снова при
следующем запуске ячейки.

In [16]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

You can directly call this tokenizer on one sentence:

In [17]:
tokenizer("Hello, this is one sentence!")

{'input_ids': [2, 9944, 16, 881, 550, 835, 15503, 5, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

Depending on the model you selected, you will see different keys in the dictionary returned by the cell above. They don't matter much for what we're doing here (just know they are required by the model we will instantiate later), you can learn more about them in [this tutorial](https://huggingface.co/transformers/preprocessing.html) if you're interested.

Так как в нашем случае входные данные уже были разделены на слова,
мы должны передать список слов в свой токенайзер с аргументом `is_split_into_words=True`:

In [18]:
tokenizer(["Hello", ",", "this", "is", "one", "sentence", "split", "into", "words", "."], is_split_into_words=True)

{'input_ids': [2, 9944, 16, 881, 550, 835, 15503, 7440, 996, 6301, 18, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Обратим внимание, что трансформаторы часто предварительно
обучаются с помощью токенизаторов вложенных слов, что означает, что даже если входные данные уже были разделены на слова, каждое из этих
слов может быть снова разделено токенизатором.

In [19]:
example = ner_train[5]
print(example["tokens"])

['Мы', 'поменяли', 'место', 'жительства', 'и', 'перевели', 'дочь', 'в', 'школу', ',', 'которая', 'находится', 'ближе', 'к', 'дому', '.']


In [20]:
tokenized_input = tokenizer(example["tokens"], is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
print(tokens)

['[CLS]', 'Мы', 'пом', '##ен', '##яли', 'место', 'ж', '##итель', '##ства', 'и', 'пер', '##еве', '##ли', 'дочь', 'в', 'школу', ',', 'которая', 'находится', 'б', '##ли', '##же', 'к', 'дому', '.', '[SEP]']


Чтобы перейти с уровня слов на уровень subword tokens, нужно ещё раз предобработать тексты.

In [21]:
len(example["tags"]), len(tokenized_input["input_ids"])

(16, 26)

К счастью, токенизатор возвращает выходные данные, которые имеют
метод `word_ids`, который может нам помочь.

In [22]:
print(tokenized_input.word_ids())

[None, 0, 1, 1, 1, 2, 3, 3, 3, 4, 5, 5, 5, 6, 7, 8, 9, 10, 11, 12, 12, 12, 13, 14, 15, None]


Как мы можем заметить, токенизатор возвращает список с тем же
количеством элементов, что и наши обработанные входные идентификаторы,
сопоставляя специальные токены с None и все остальные токены с их
соответствующим словом. Таким образом, мы можем выровнять метки с
обработанными входными идентификаторами.

In [23]:
word_ids = tokenized_input.word_ids()
aligned_labels = [-100 if i is None else example["tags"][i] for i in word_ids]
print(len(aligned_labels), len(tokenized_input["input_ids"]))

26 26


Здесь мы устанавливаем метки всех специальных токенов равными -
100 (индекс, который игнорируется PyTorch), а метки всех остальных токенов – метке слова, из которого они происходят. Другая стратегия заключается в том, чтобы установить метку только для первого токена, полученного из данного слова, и присвоить метку -100 другим подтокенам из того же слова. Мы предлагаем здесь две стратегии, просто измените флаг `label_all_tokens`.

Теперь мы готовы написать функцию, которая будет предварительно
обрабатывать наши образцы. Мы отправляем их в токенизатор с аргументом
truncation=True (для усечения текстов, размер которых превышает
максимальный размер, разрешенный моделью) и is_split_into_words=True
(как показано выше). Затем мы выравниваем метки с идентификаторами
токенов, используя выбранную нами стратегию:

In [24]:
def tokenize_and_align_labels(examples, label_all_tokens=True):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples['tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        label_ids = [label_list.index(idx) if isinstance(idx, str) else idx for idx in label_ids]

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

Эта функция работает с одним или несколькими примерами. В случае
нескольких примеров токенизатор вернет список для каждого ключа:

In [25]:
tokenize_and_align_labels(ner_data['train'][22:23])

{'input_ids': [[2, 1041, 4033, 3236, 9267, 331, 19173, 19106, 26629, 1887, 22018, 548, 22276, 320, 21538, 16, 705, 13718, 22264, 548, 18397, 14063, 11137, 626, 16296, 24531, 18, 3]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [[-100, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100]]}

Чтобы применить эту функцию ко всем предложениям (или парам
предложений) в нашем наборе данных, мы просто используем метод map
нашего объекта dataset, который мы создали ранее. Это применит функцию
ко всем элементам всех разбиений в dataset, так что наши обучающие,
валидационные и тестовые данные будут предварительно обработаны одной
командой.

In [26]:
tokenized_datasets = ner_data.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/3847 [00:00<?, ? examples/s]

Map:   0%|          | 0/962 [00:00<?, ? examples/s]

Результаты автоматически кэшируются библиотекой наборов данных, что позволяет не тратить время на этот шаг при следующем запуске. Библиотека Datasets обычно способна определить, когда функция, передаваемая в `map`, изменилась (и, следовательно, требует не использовать данные кэша). Datasets предупреждает вас, когда использует кэшированные файлы, можно передать `load_from_cache_file=False` в вызове `map`, чтобы не использовать кэшированные файлы и принудительно применить предварительную обработку снова.

Обратите внимание, что передали `batched=True` для совместного кодирования текстов пакетами. Это сделано для того, чтобы в полной мере использовать преимущества быстрого токенизатора, который был загружен ранее, который будет использовать многопоточность для одновременной обработки текстов в пакете.

## Fine-tuning the model

После подготовки данных можно загрузить предварительно подготовленную модель и точно настроить ее. Поскольку задачи связаны с классификацией токенов, используем класс `AutoModelForTokenClassification`. Как и в случае с токенизатором, метод `from_pretrained` загрузит и кэширует модель. Единственное, что требуется указать, это количество меток для задачи (которые можно получить из функций, как показано ранее):

In [27]:
label_list

['O',
 'B-ADR',
 'B-DI',
 'B-Drugclass',
 'B-Drugform',
 'B-Drugname',
 'B-Finding',
 'I-ADR',
 'I-DI',
 'I-Drugclass',
 'I-Drugform',
 'I-Drugname',
 'I-Finding']

In [28]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list))
model.config.id2label = dict(enumerate(label_list))
model.config.label2id = {v: k for k, v in model.config.id2label.items()}

model.safetensors:   0%|          | 0.00/47.7M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Предупреждение говорит о том, что мы отбрасываем некоторые веса
(слои `vocab_transform` и vocab_layer_norm) и случайным образом
инициализируем некоторые другие (слои `pre_classifier` и `classifier`). В данном
случае это абсолютно нормально, потому что мы удаляем head, используемый для предварительной обработки модели в задаче моделирования на замаскированном языке, и заменяем его новым head, для которого у нас нет предварительно обработанных весов, поэтому библиотека предупреждает нас, что мы должны точно настроить эту модель, прежде чем использовать ее для вывода.

Чтобы создать экземпляр `Trainer`, нужно определить еще три
вещи. Наиболее важным является [`TrainingArguments`](https://huggingface.co/transformers/main_classes/trainer.html#transformers.TrainingArguments), который представляет собой класс, содержащий все атрибуты для настройки обучения. Для этого требуется имя папки, которое будет использоваться для сохранения
контрольных точек модели, а все остальные аргументы необязательны:

In [29]:
args = TrainingArguments(
    "ner",
    eval_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=10,
    weight_decay=0.01,
    save_strategy='no',
    report_to='none',
)

Далее устанавливаем оценку, которая будет выполняться в конце
каждой эпохи, настраиваем скорость обучения, используем `batch_size`,
определенный в верхней части блокнота, и настраиваем количество эпох для
обучения, а также уменьшение веса.

Также понадобится средство сбора данных, которое будет группировать обработанные примеры вместе, применяя отступы, чтобы сделать их все одинакового размера (каждый блокнот будет дополнен до длины самого длинного примера). Для этой задачи в библиотеке Transformers есть сборщик данных, который передает не только входные данные, но и метки.

In [30]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

И последнее, что нужно определить для тренировочной выборки
– это как вычислять показатели на основе прогнозов. Для этого загрузим
метрику [`seqeval`](https://github.com/chakki-works/seqeval) (которая обычно используется для оценки результатов в наборе данных CONLL) через библиотеку Datasets.

In [31]:
!pip install evaluate
import evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [32]:
metric = evaluate.load("seqeval")

Эта метрика принимает список меток для прогнозов и ссылок:

In [33]:
example = ner_train[4]
labels = example['tags']
metric.compute(predictions=[labels], references=[labels])

{'DI': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(1)},
 'Drugform': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(2)},
 'overall_precision': np.float64(1.0),
 'overall_recall': np.float64(1.0),
 'overall_f1': np.float64(1.0),
 'overall_accuracy': 1.0}

Поэтому необходимо провести небольшую постобработку предсказаний:
- выбираем прогнозируемый индекс (с максимальным логитом) для
каждого токена;
- преобразуем его в строковую метку;
- игнорируем везде, где мы устанавливаем метку -100.

Следующая функция выполняет всю эту постобработку результата
`Trainer.evaluate` (который представляет собой `namedtuple`, содержащий
прогнозы и метки) перед применением метрики.

In [34]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels, zero_division=0)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Обратим внимание, что мы отбрасываем precision/recall/f1,
вычисленные для каждой категории, и фокусируемся только на общих
precision/recall/f1.
Затем просто нужно передать все это вместе с наборами данных тренеру `Trainer`:

In [35]:
trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-2765119595.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [41]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 2.682187795639038,
 'eval_model_preparation_time': 0.002,
 'eval_precision': 0.023355642741749366,
 'eval_recall': 0.15436241610738255,
 'eval_f1': 0.040572505312111616,
 'eval_accuracy': 0.05137469889525708,
 'eval_runtime': 20.4464,
 'eval_samples_per_second': 47.05,
 'eval_steps_per_second': 2.983}

В начале обучения заморозим все параметры в модели, кроме последнего слоя, и посмотрим, насколько хорошо она обучится.

In [ ]:
for param in model.bert.parameters():
    param.requires_grad = False

In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)
        print(param)

classifier.weight
Parameter containing:
tensor([[-5.3295e-02,  8.1591e-05, -1.4091e-02,  ...,  9.4435e-03,
          2.6371e-02, -2.7459e-02],
        [-1.4154e-02,  1.8980e-02, -6.4149e-03,  ..., -3.0063e-02,
         -8.0335e-03, -1.3474e-02],
        [ 3.9226e-03, -1.7339e-03, -2.4043e-03,  ...,  1.1911e-02,
         -6.8623e-03, -3.6764e-02],
        ...,
        [ 2.9699e-02, -2.5830e-02,  2.9956e-03,  ...,  2.0724e-02,
          2.6304e-02, -1.3127e-04],
        [-2.8258e-02,  1.9521e-03, -1.2629e-02,  ..., -2.4292e-02,
         -1.9133e-02,  3.5226e-02],
        [ 4.8563e-03, -3.9019e-02,  2.2573e-02,  ...,  2.3094e-02,
         -5.4334e-03, -3.1281e-02]], device='cuda:0', requires_grad=True)
classifier.bias
Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0',
       requires_grad=True)


Теперь можно точно настроить модель, просто вызвав метод `train`:

In [ ]:
import logging
from transformers.trainer import logger as noisy_logger
noisy_logger.setLevel(logging.WARNING)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,2.034866,0.032157,0.059487,0.041747,0.630991
2,No log,1.594469,0.042105,0.004881,0.008748,0.815724
3,2.043100,1.282439,0.052632,0.000305,0.000607,0.826314
4,2.043100,1.079169,0.000000,0.000000,0.000000,0.826854
5,1.267200,0.954540,0.000000,0.000000,0.000000,0.826896
6,1.267200,0.880644,0.000000,0.000000,0.000000,0.826896
7,0.932500,0.837882,0.000000,0.000000,0.000000,0.826896
8,0.932500,0.813664,0.000000,0.000000,0.000000,0.826896
9,0.808700,0.801121,0.000000,0.000000,0.000000,0.826896
10,0.808700,0.797258,0.000000,0.000000,0.000000,0.826896


TrainOutput(global_step=2410, training_loss=1.181212188594074, metrics={'train_runtime': 31.5523, 'train_samples_per_second': 1219.246, 'train_steps_per_second': 76.381, 'total_flos': 35752217175750.0, 'train_loss': 1.181212188594074, 'epoch': 10.0})

Модель недообучилась, вероятно, нужно обучить больше слоёв.

Чтобы получить precision/recall/f1, вычисленную для каждой категории, можно применить ту же функцию, что и раньше, к результату метода `predict`:

In [ ]:
predictions, labels, _ = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

results = metric.compute(predictions=true_predictions, references=true_labels)
results

/usr/local/lib/python3.7/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'ADR': {'f1': 0.30279898218829515,
  'number': 446,
  'precision': 0.35,
  'recall': 0.26681614349775784},
 'DI': {'f1': 0.493963782696177,
  'number': 821,
  'precision': 0.4207369323050557,
  'recall': 0.5980511571254568},
 'Drugclass': {'f1': 0.7868852459016393,
  'number': 336,
  'precision': 0.7880597014925373,
  'recall': 0.7857142857142857},
 'Drugform': {'f1': 0.7922794117647058,
  'number': 565,
  'precision': 0.8240917782026769,
  'recall': 0.7628318584070797},
 'Drugname': {'f1': 0.8734309623430963,
  'number': 918,
  'precision': 0.8400402414486922,
  'recall': 0.9095860566448801},
 'Finding': {'f1': 0.0, 'number': 192, 'precision': 0.0, 'recall': 0.0},
 'overall_accuracy': 0.9050170279923582,
 'overall_f1': 0.6448696700316409,
 'overall_precision': 0.6370943733253944,
 'overall_recall': 0.652837095790116}

In [36]:
from sklearn.metrics import confusion_matrix
import pandas as pd

In [ ]:
cm = pd.DataFrame(
    confusion_matrix(sum(true_labels, []), sum(true_predictions, []), labels=label_list),
    index=label_list,
    columns=label_list
)
cm

,O,B-ADR,B-DI,B-Drugclass,B-Drugform,B-Drugname,B-Finding,I-ADR,I-DI,I-Drugclass,I-Drugform,I-Drugname,I-Finding
O,19494,29,175,35,60,71,0,20,26,0,0,0,0
B-ADR,159,135,133,8,2,0,0,4,5,0,0,0,0
B-DI,242,21,525,0,17,10,0,3,3,0,0,0,0
B-Drugclass,50,1,17,264,0,4,0,0,0,0,0,0,0
B-Drugform,98,4,11,1,432,17,0,1,1,0,0,0,0
B-Drugname,44,1,16,1,8,848,0,0,0,0,0,0,0
B-Finding,56,32,87,5,3,3,0,1,5,0,0,0,0
I-ADR,180,51,40,0,1,0,0,47,30,0,0,0,0
I-DI,236,17,102,10,0,1,0,11,46,0,0,0,0
I-Drugclass,0,0,0,4,0,0,0,0,0,0,0,0,0


In [ ]:
model.save_pretrained('ner_bert.bin')
tokenizer.save_pretrained('ner_bert.bin')

Configuration saved in ner_bert.bin/config.json
Model weights saved in ner_bert.bin/pytorch_model.bin
tokenizer config file saved in ner_bert.bin/tokenizer_config.json
Special tokens file saved in ner_bert.bin/special_tokens_map.json


('ner_bert.bin/tokenizer_config.json',
 'ner_bert.bin/special_tokens_map.json',
 'ner_bert.bin/vocab.txt',
 'ner_bert.bin/added_tokens.json',
 'ner_bert.bin/tokenizer.json')

# Задание

**Задание**. Модель недообучилась, вероятно, нужно обучить больше слоёв.
Разморозьте несколько верхних или все слои (param.requires_grad = True) и дообучите модель. При этом, возможно, потребуется увеличить количество эпох обучения. Можно также поэспериментировать с другими гиперпараметрами (batch_size, learning_rate и др.)

## Разморозим все слои

In [42]:
for param in model.bert.parameters():
    param.requires_grad = True

In [48]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)
        print(param)

bert.embeddings.word_embeddings.weight
Parameter containing:
tensor([[ 0.0006, -0.0328, -0.0623,  ..., -0.0355, -0.0559, -0.0132],
        [ 0.0959, -0.0789, -0.0168,  ..., -0.0862, -0.0831, -0.0013],
        [ 0.0060, -0.0554,  0.0143,  ..., -0.0006,  0.0040,  0.0125],
        ...,
        [-0.0306, -0.1518, -0.1168,  ...,  0.0059, -0.0196, -0.0278],
        [ 0.0520, -0.1090,  0.0432,  ..., -0.0089, -0.1032, -0.0405],
        [ 0.0057, -0.2035, -0.0105,  ...,  0.0129,  0.0341, -0.0240]],
       requires_grad=True)
bert.embeddings.position_embeddings.weight
Parameter containing:
tensor([[-0.0101,  0.0034,  0.0084,  ...,  0.0159,  0.0042, -0.0047],
        [-0.0154,  0.0136,  0.0322,  ..., -0.0268,  0.0189,  0.0112],
        [-0.0157,  0.0082,  0.0333,  ..., -0.0183,  0.0189,  0.0024],
        ...,
        [ 0.0158,  0.0536, -0.0011,  ...,  0.0021, -0.0024,  0.0041],
        [ 0.0124,  0.0313, -0.0068,  ...,  0.0091, -0.0067,  0.0017],
        [ 0.0153,  0.0755,  0.0060,  ...,  0.0216,

In [37]:
args = TrainingArguments(
    "ner_unfrozen",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    save_strategy="no",
    report_to="none",
)

In [38]:
unfrozen_trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-2000016826.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  unfrozen_trainer = Trainer(


In [39]:
import logging
from transformers.trainer import logger as noisy_logger
noisy_logger.setLevel(logging.WARNING)

In [51]:
unfrozen_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.527736,0.685547,0.321232,0.437474,0.866766
2,No log,0.422437,0.574104,0.542404,0.557804,0.885871
3,0.609500,0.384691,0.570801,0.634533,0.600982,0.894053
4,0.609500,0.346921,0.667092,0.637584,0.652004,0.905432
5,0.338400,0.327098,0.639041,0.675107,0.656579,0.907135
6,0.338400,0.317679,0.635958,0.694936,0.664140,0.909544
7,0.266700,0.313178,0.649590,0.700122,0.673910,0.911579
8,0.266700,0.311099,0.634159,0.718121,0.673534,0.910748
9,0.221500,0.303485,0.668094,0.714155,0.690357,0.915109
10,0.221500,0.309668,0.645049,0.737340,0.688114,0.913656


TrainOutput(global_step=3615, training_loss=0.27801496682490223, metrics={'train_runtime': 2109.802, 'train_samples_per_second': 27.351, 'train_steps_per_second': 1.713, 'total_flos': 53461300581990.0, 'train_loss': 0.27801496682490223, 'epoch': 15.0})

In [52]:
predictions, labels, _ = unfrozen_trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

results = metric.compute(predictions=true_predictions, references=true_labels)
results

{'ADR': {'precision': np.float64(0.38061041292639136),
  'recall': np.float64(0.47533632286995514),
  'f1': np.float64(0.42273180458624127),
  'number': np.int64(446)},
 'DI': {'precision': np.float64(0.502360717658168),
  'recall': np.float64(0.6479902557856273),
  'f1': np.float64(0.5659574468085107),
  'number': np.int64(821)},
 'Drugclass': {'precision': np.float64(0.8652694610778443),
  'recall': np.float64(0.8601190476190477),
  'f1': np.float64(0.8626865671641791),
  'number': np.int64(336)},
 'Drugform': {'precision': np.float64(0.8810344827586207),
  'recall': np.float64(0.904424778761062),
  'f1': np.float64(0.8925764192139739),
  'number': np.int64(565)},
 'Drugname': {'precision': np.float64(0.8524426719840479),
  'recall': np.float64(0.9313725490196079),
  'f1': np.float64(0.8901613742842269),
  'number': np.int64(918)},
 'Finding': {'precision': np.float64(0.48),
  'recall': np.float64(0.0625),
  'f1': np.float64(0.11059907834101382),
  'number': np.int64(192)},
 'overall

Видим, что по сравнению с замороженной версией overall_f1 и overall_recall выросли почти на 10%, overall_precision почти на 5%

In [64]:
from sklearn.metrics import confusion_matrix
import pandas as pd

In [54]:
cm = pd.DataFrame(
    confusion_matrix(sum(true_labels, []), sum(true_predictions, []), labels=label_list),
    index=label_list,
    columns=label_list
)
cm

,O,B-ADR,B-DI,B-Drugclass,B-Drugform,B-Drugname,B-Finding,I-ADR,I-DI,I-Drugclass,I-Drugform,I-Drugname,I-Finding
O,19379,76,192,25,49,84,3,25,77,0,0,0,0
B-ADR,105,235,83,1,0,0,8,13,1,0,0,0,0
B-DI,161,58,557,0,12,7,0,8,18,0,0,0,0
B-Drugclass,32,0,15,289,0,0,0,0,0,0,0,0,0
B-Drugform,45,0,2,2,512,4,0,0,0,0,0,0,0
B-Drugname,36,2,4,1,3,870,0,1,1,0,0,0,0
B-Finding,39,68,63,0,3,2,14,2,1,0,0,0,0
I-ADR,132,48,15,0,0,0,0,104,50,0,0,0,0
I-DI,161,26,58,5,0,0,0,17,156,0,0,0,0
I-Drugclass,0,0,0,4,0,0,0,0,0,0,0,0,0


In [55]:
model.save_pretrained('ner_bert.bin')
tokenizer.save_pretrained('ner_bert.bin')

('ner_bert.bin/tokenizer_config.json',
 'ner_bert.bin/special_tokens_map.json',
 'ner_bert.bin/vocab.txt',
 'ner_bert.bin/added_tokens.json',
 'ner_bert.bin/tokenizer.json')

## Подбор гиперпараметров

In [40]:
!pip install -q optuna
import optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 5.3 MB/s eta 0:00:00


In [41]:
def model_init():
  model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list))
  model.config.id2label = dict(enumerate(label_list))
  model.config.label2id = {v: k for k, v in model.config.id2label.items()}
  for param in model.bert.parameters():
    param.requires_grad = True
  return model

In [42]:
def hp_space(trial: optuna.trial.Trial):
    return {
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-5, 5e-4),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 10),
        "weight_decay": trial.suggest_categorical("weight_decay", [0.0, 0.01, 0.1])
    }

In [45]:
hp_args = TrainingArguments(
    "ner_unfrozen",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    save_strategy="no",
    report_to="none",
    disable_tqdm=True
)

In [46]:
hp_serch_trainer = Trainer(
    model_init=model_init,
    args=hp_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


/tmp/ipython-input-1688016004.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  hp_serch_trainer = Trainer(
Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [47]:
hp_search = hp_serch_trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    n_trials=20,
    hp_space=hp_space,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2, interval_steps=1)
)

[I 2025-12-18 09:24:57,788] A new study created in memory with name: no-name-2d8f4c88-08e1-4d2c-8fac-fe8ebe258a6d
/tmp/ipython-input-18836203.py:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate": trial.suggest_loguniform("learning_rate", 1e-5, 5e-4),
Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.34286364912986755, 'eval_precision': 0.6971252566735113, 'eval_recall': 0.6214154972544235, 'eval_f1': 0.6570967741935484, 'eval_accuracy': 0.9047678378602874, 'eval_runtime': 9.5899, 'eval_samples_per_second': 100.314, 'eval_steps_per_second': 6.361, 'epoch': 1.0}
{'eval_loss': 0.2938852310180664, 'eval_precision': 0.6377388535031847, 'eval_recall': 0.7330689444783405, 'eval_f1': 0.6820891285835935, 'eval_accuracy': 0.9137386826148352, 'eval_runtime': 8.8628, 'eval_samples_per_second': 108.543, 'eval_steps_per_second': 6.883, 'epoch': 2.0}
{'eval_loss': 0.2821999788284302, 'eval_precision': 0.6608465608465608, 'eval_recall': 0.7620500305064063, 'eval_f1': 0.7078492490790591, 'eval_accuracy': 0.9193869922751059, 'eval_runtime': 8.6925, 'eval_samples_per_second': 110.67, 'eval_steps_per_second': 7.018, 'epoch': 3.0}
{'eval_loss': 0.2888328433036804, 'eval_precision': 0.7297297297297297, 'eval_recall': 0.7330689444783405, 'eval_f1': 0.7313955257951607, 'eval_accuracy': 0.

[I 2025-12-18 09:48:37,198] Trial 0 finished with value: 3.172454805701648 and parameters: {'learning_rate': 0.00024861549613199355, 'per_device_train_batch_size': 32, 'num_train_epochs': 8, 'weight_decay': 0.1}. Best is trial 0 with value: 3.172454805701648.
Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'eval_loss': 0.39438462257385254, 'eval_precision': 0.6383350821965722, 'eval_recall': 0.5567419158023185, 'eval_f1': 0.5947531367117485, 'eval_accuracy': 0.8938865354265304, 'eval_runtime': 9.4068, 'eval_samples_per_second': 102.267, 'eval_steps_per_second': 6.485, 'epoch': 1.0}
{'eval_loss': 0.32395556569099426, 'eval_precision': 0.6214848143982002, 'eval_recall': 0.6741915802318487, 'eval_f1': 0.6467661691542288, 'eval_accuracy': 0.907342802558352, 'eval_runtime': 9.3056, 'eval_samples_per_second': 103.378, 'eval_steps_per_second': 6.555, 'epoch': 2.0}
{'eval_loss': 0.31172239780426025, 'eval_precision': 0.6499429874572406, 'eval_recall': 0.6955460646735815, 'eval_f1': 0.671971706454465, 'eval_accuracy': 0.9129911122186228, 'eval_runtime': 9.5671, 'eval_samples_per_second': 100.552, 'eval_steps_per_second': 6.376, 'epoch': 3.0}
{'train_runtime': 498.719, 'train_samples_per_second': 23.141, 'train_steps_per_second': 0.728, 'train_loss': 0.39601492553374656, 'epoch': 3.0}


[I 2025-12-18 09:56:57,511] Trial 1 finished with value: 2.93045187080391 and parameters: {'learning_rate': 0.00015720597738360894, 'per_device_train_batch_size': 32, 'num_train_epochs': 3, 'weight_decay': 0.0}. Best is trial 0 with value: 3.172454805701648.
Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

{'eval_loss': 0.47032424807548523, 'eval_precision': 0.5859604875998319, 'eval_recall': 0.42525930445393534, 'eval_f1': 0.4928407283012198, 'eval_accuracy': 0.8756125924080073, 'eval_runtime': 9.0992, 'eval_samples_per_second': 105.723, 'eval_steps_per_second': 6.704, 'epoch': 1.0}
{'eval_loss': 0.31568536162376404, 'eval_precision': 0.6393351800554017, 'eval_recall': 0.7040878584502746, 'eval_f1': 0.670150987224158, 'eval_accuracy': 0.9105407425865936, 'eval_runtime': 9.3772, 'eval_samples_per_second': 102.589, 'eval_steps_per_second': 6.505, 'epoch': 1.0}
{'loss': 0.4264, 'grad_norm': 3.021831750869751, 'learning_rate': 0.00022435957083682956, 'epoch': 1.0395010395010396}
{'eval_loss': 0.27813759446144104, 'eval_precision': 0.6899661781285231, 'eval_recall': 0.7467968273337401, 'eval_f1': 0.7172575446820979, 'eval_accuracy': 0.9215466400863859, 'eval_runtime': 9.648, 'eval_samples_per_second': 99.709, 'eval_steps_per_second': 6.323, 'epoch': 2.0}
{'loss': 0.2077, 'grad_norm': 5.13601

[I 2025-12-18 12:20:24,414] Trial 17 finished with value: 3.1582774490016328 and parameters: {'learning_rate': 0.0002830979894347193, 'per_device_train_batch_size': 8, 'num_train_epochs': 5, 'weight_decay': 0.1}. Best is trial 0 with value: 3.172454805701648.
Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[I 2025-12-18 12:22:36,375] Trial 18 pruned. 
Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'eval_loss': 0.33018848299980164, 'eval_precision': 0.6383593298671288, 'eval_recall': 0.6741915802318487, 'eval_f1': 0.655786350148368, 'eval_accuracy': 0.9066782955394966, 'eval_runtime': 8.7697, 'eval_samples_per_second': 109.695, 'eval_steps_per_second': 6.956, 'epoch': 1.0}


[I 2025-12-18 12:25:25,368] Trial 19 pruned. 


{'eval_loss': 0.5176547169685364, 'eval_precision': 0.6170212765957447, 'eval_recall': 0.32733374008541793, 'eval_f1': 0.4277456647398844, 'eval_accuracy': 0.8652296702383919, 'eval_runtime': 9.6676, 'eval_samples_per_second': 99.507, 'eval_steps_per_second': 6.31, 'epoch': 1.0}


In [48]:
print(hp_search)

BestRun(run_id='0', objective=3.172454805701648, hyperparameters={'learning_rate': 0.00024861549613199355, 'per_device_train_batch_size': 32, 'num_train_epochs': 8, 'weight_decay': 0.1}, run_summary=None)


### Best model

In [53]:
from transformers import EarlyStoppingCallback

In [61]:
best_model = model_init()

best_args = TrainingArguments(
    "ner_unfrozen",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=0.00024861549613199355,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=15,
    weight_decay=0.1,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

best_trainer = Trainer(
    model=best_model,
    args=best_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2443800760.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  best_trainer = Trainer(


In [62]:
best_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.352031,0.702607,0.608298,0.652060,0.902401
2,No log,0.290399,0.657104,0.740696,0.696400,0.914279
3,No log,0.279439,0.659346,0.769372,0.710122,0.919636
4,No log,0.290287,0.745920,0.711104,0.728096,0.926447
5,0.266500,0.309921,0.721134,0.752593,0.736528,0.926281
6,0.266500,0.337896,0.688634,0.752288,0.719055,0.922502
7,0.266500,0.364707,0.745993,0.752593,0.749279,0.928109
8,0.266500,0.362933,0.714853,0.769372,0.741111,0.928316
9,0.048400,0.388641,0.750310,0.738865,0.744543,0.929313


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument i

TrainOutput(global_step=1089, training_loss=0.14590463537596254, metrics={'train_runtime': 1559.2689, 'train_samples_per_second': 37.008, 'train_steps_per_second': 1.164, 'total_flos': 38281983441150.0, 'train_loss': 0.14590463537596254, 'epoch': 9.0})

In [63]:
predictions, labels, _ = best_trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

results = metric.compute(predictions=true_predictions, references=true_labels)
results

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'ADR': {'precision': np.float64(0.4966139954853273),
  'recall': np.float64(0.49327354260089684),
  'f1': np.float64(0.49493813273340825),
  'number': np.int64(446)},
 'DI': {'precision': np.float64(0.608843537414966),
  'recall': np.float64(0.6540803897685749),
  'f1': np.float64(0.6306517909571344),
  'number': np.int64(821)},
 'Drugclass': {'precision': np.float64(0.8832335329341318),
  'recall': np.float64(0.8779761904761905),
  'f1': np.float64(0.8805970149253731),
  'number': np.int64(336)},
 'Drugform': {'precision': np.float64(0.8756218905472637),
  'recall': np.float64(0.9345132743362832),
  'f1': np.float64(0.904109589041096),
  'number': np.int64(565)},
 'Drugname': {'precision': np.float64(0.9130434782608695),
  'recall': np.float64(0.9150326797385621),
  'f1': np.float64(0.914036996735582),
  'number': np.int64(918)},
 'Finding': {'precision': np.float64(0.376),
  'recall': np.float64(0.24479166666666666),
  'f1': np.float64(0.29652996845425866),
  'number': np.int64(192)

In [65]:
cm = pd.DataFrame(
    confusion_matrix(sum(true_labels, []), sum(true_predictions, []), labels=label_list),
    index=label_list,
    columns=label_list
)
cm

,O,B-ADR,B-DI,B-Drugclass,B-Drugform,B-Drugname,B-Finding,I-ADR,I-DI,I-Drugclass,I-Drugform,I-Drugname,I-Finding
O,19477,53,123,24,59,53,30,28,51,0,0,3,9
B-ADR,120,230,71,0,0,0,9,12,4,0,0,0,0
B-DI,175,38,557,0,11,2,20,4,11,0,0,0,3
B-Drugclass,30,0,6,295,0,0,5,0,0,0,0,0,0
B-Drugform,34,0,2,0,529,0,0,0,0,0,0,0,0
B-Drugname,54,1,4,0,3,848,3,0,3,0,0,2,0
B-Finding,50,42,50,0,1,0,47,2,0,0,0,0,0
I-ADR,118,35,4,0,0,0,0,157,35,0,0,0,0
I-DI,164,14,34,4,0,0,4,29,166,0,0,0,8
I-Drugclass,0,0,0,4,0,0,0,0,0,0,0,0,0


### Вывод

Получилось улучшить метрики:

**overall_precision**: с $0.637$ до $0.746$ $\approx +11\%$

**overall_recall**: с $0.653$ до $0.753$ $\approx +10\%$

**overall_f1**: с $0.645$ до $0.749$ $\approx +10\%$

**overall_accuracy**: с $0.905$ до $0.928$ $\approx +2\%$

In [66]:
best_model.save_pretrained('best_ner_bert.bin')
tokenizer.save_pretrained('best_ner_bert.bin')

('best_ner_bert.bin/tokenizer_config.json',
 'best_ner_bert.bin/special_tokens_map.json',
 'best_ner_bert.bin/vocab.txt',
 'best_ner_bert.bin/added_tokens.json',
 'best_ner_bert.bin/tokenizer.json')

### Тест

In [76]:
from transformers import AutoTokenizer, AutoModel

In [77]:
tokenizer = AutoTokenizer.from_pretrained('best_ner_bert.bin')
model = AutoModel.from_pretrained(
    'best_ner_bert.bin',
    output_attentions=True
)

model.eval()

Some weights of BertModel were not initialized from the model checkpoint at best_ner_bert.bin and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(29564, 312, padding_idx=0)
    (position_embeddings): Embedding(512, 312)
    (token_type_embeddings): Embedding(2, 312)
    (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-2): 3 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=312, out_features=312, bias=True)
            (key): Linear(in_features=312, out_features=312, bias=True)
            (value): Linear(in_features=312, out_features=312, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=312, out_features=312, bias=True)
            (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
    

In [90]:
text = " ".join(ner_test[17]["tokens"])
text

'И вот на третий день приема тонзилгона к вечеру у него появились красненькие пятнышки на спинке и личике , а ночью он был весь красный .'

In [91]:
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True
)

In [71]:
!pip install -q bertviz
import torch
from bertviz import head_view, model_view

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.1 MB/s eta 0:00:00


In [92]:
with torch.no_grad():
    outputs = model(**inputs)

In [93]:
attentions = outputs.attentions

In [94]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

head_view(
    attentions,
    tokens
)

Output hidden; open in https://colab.research.google.com to view.

In [95]:
model_view(
    attentions,
    tokens
)

Output hidden; open in https://colab.research.google.com to view.